# ⚡ Day 05: Advanced Stack & Queue Patterns

---

## The Monotonic Deque — Best of Both Worlds

We saw monotonic stacks for "next greater" problems. A **monotonic deque** is the same idea but allows removal from both ends — perfect for sliding window problems where old elements need to expire.

This pattern solves one of the most important hard problems:

---

## 🎯 Problem 1: Sliding Window Maximum (LC #239)

Given an array and window size k, return the max in each window as it slides.

```
Input:  nums = [1,3,-1,-3,5,3,6,7], k = 3
Output: [3,3,5,5,6,7]

Window [1, 3,-1] → max=3
Window [3,-1,-3] → max=3
Window [-1,-3, 5] → max=5
...and so on
```

Brute force: O(nk) — check all k elements per window. Can we do O(n)?

**Monotonic deque approach:** Maintain a deque of indices where values are in decreasing order. The front always holds the current max. When the window slides:
1. Remove from front if it's outside the window
2. Remove from back anything smaller than the new element (they'll never be the max)
3. Add new element to back
4. Front is the answer

In [ ]:
from collections import deque

def maxSlidingWindow(nums, k):
    """
    Monotonic deque: decreasing order of values, stores indices.
    Time: O(n), Space: O(k)
    """
    dq = deque()   # Indices, values in decreasing order
    result = []
    
    for i in range(len(nums)):
        # 1. Remove expired index from front
        if dq and dq[0] < i - k + 1:
            dq.popleft()
        
        # 2. Remove smaller elements from back (they're useless)
        while dq and nums[dq[-1]] < nums[i]:
            dq.pop()
        
        # 3. Add current index
        dq.append(i)
        
        # 4. Record answer once first window is complete
        if i >= k - 1:
            result.append(nums[dq[0]])  # Front = max
    
    return result

nums = [1, 3, -1, -3, 5, 3, 6, 7]
print(f"Input:  {nums}, k=3")
print(f"Output: {maxSlidingWindow(nums, 3)}")  # [3,3,5,5,6,7]

# Trace the deque state
print("\nTrace:")
dq = deque()
for i in range(len(nums)):
    if dq and dq[0] < i - 3 + 1:
        dq.popleft()
    while dq and nums[dq[-1]] < nums[i]:
        dq.pop()
    dq.append(i)
    if i >= 2:
        print(f"  Window [{i-2}:{i+1}] = {nums[i-2:i+1]}, deque indices={list(dq)}, max={nums[dq[0]]}")

**Why is this O(n)?** Every index enters the deque once and leaves at most once. The while loop looks dangerous, but across all iterations, it does at most n pops total.

---

## 🎯 Problem 2: Next Greater Element II — Circular Array (LC #503)

Same as Next Greater Element but the array is circular — after the last element, it wraps back to the first.

```
Input:  [1, 2, 1]
Output: [2, -1, 2]   (for 1 at index 2, wrapping around finds 2 at index 1)
```

**The trick:** Process the array twice (2n iterations) using `i % n` to simulate the circular wrap. Only push indices from the first pass.

In [ ]:
def nextGreaterElements(nums):
    """
    Circular next greater using double traversal.
    Time: O(n), Space: O(n)
    """
    n = len(nums)
    result = [-1] * n
    stack = []  # Stack of indices
    
    # Process array twice to handle circular
    for i in range(2 * n):
        actual_i = i % n
        while stack and nums[actual_i] > nums[stack[-1]]:
            result[stack.pop()] = nums[actual_i]
        # Only push indices from the first pass
        if i < n:
            stack.append(actual_i)
    
    return result

print(nextGreaterElements([1, 2, 1]))        # [2, -1, 2]
print(nextGreaterElements([1, 2, 3, 4, 3]))  # [2, 3, 4, -1, 4]

---

## 🎯 Problem 3: Online Stock Span (LC #901)

For each day's stock price, find the span — how many consecutive previous days had price ≤ today's price.

```
Prices: [100, 80, 60, 70, 60, 75, 85]
Spans:  [1,   1,  1,  2,  1,  4,  6]

For 75: prices 60, 70, 60, 75 — span = 4
For 85: prices 60, 75, 60, 70, 60, 85 — span = 6
```

**Monotonic decreasing stack of (price, span) pairs.** When we pop elements, we absorb their spans.

In [ ]:
class StockSpanner:
    def __init__(self):
        self.stack = []  # (price, span)
    
    def next(self, price):
        span = 1
        # Pop and absorb spans of smaller prices
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span

spanner = StockSpanner()
prices = [100, 80, 60, 70, 60, 75, 85]
for p in prices:
    print(f"  Price {p:3d} → span = {spanner.next(p)}")

**Why store spans instead of counting?** Because a single pop can represent many days. When we pop `(70, 2)`, that 2 tells us "70 already covered 2 days" — we absorb all of them.

---

## 🗺️ Advanced Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  ADVANCED STACK/QUEUE PATTERNS                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. MONOTONIC DEQUE (sliding window max/min):                       ║
║     Deque of indices, decreasing values for max, increasing for min ║
║     Remove expired from front, remove useless from back             ║
║                                                                      ║
║  2. CIRCULAR ARRAY trick: iterate 2n times, use i % n              ║
║     Works for next greater circular, circular subarray, etc.        ║
║                                                                      ║
║  3. SPAN / ABSORPTION: Stack stores (value, accumulated_count)     ║
║     On pop, absorb the popped element's count into current          ║
║                                                                      ║
║  DECISION GUIDE:                                                     ║
║  • Fixed window + extremum → Monotonic deque                        ║
║  • Next greater + circular → Double iteration                       ║
║  • Span / lookback count  → Stack with absorption                   ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 239 | Sliding Window Maximum | Hard | Monotonic deque |
| 503 | Next Greater Element II | Medium | Circular + monotonic stack |
| 901 | Online Stock Span | Medium | Stack with span absorption |
| 862 | Shortest Subarray with Sum at Least K | Hard | Monotonic deque + prefix sum |
| 1438 | Longest Subarray with Abs Diff ≤ Limit | Medium | Two monotonic deques (max + min) |
| 316 | Remove Duplicate Letters | Medium | Monotonic stack + greedy |

---

## ✅ Chapter 03 Final Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Stack: matching (parens), monotonic (next greater), eval       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Monotonic deque = monotonic stack + expiry from front          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Sliding window max/min → monotonic deque in O(n)              │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Circular arrays → iterate 2n times with i % n                 │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Queue for BFS. Stack for DFS. This never changes.             │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Chapter: Trees!

Binary trees, BSTs, traversals, and the recursive thinking that powers everything.

---

### 🎉 Chapter 03 Complete — Stacks & Queues Mastered!
From basic LIFO/FIFO to monotonic deques — you're ready for trees.